In [16]:
"""
Phase 1 & 2: Data download, walk-forward engine, and three VaR/ES models.
SPY 2005-2024 | HS, GARCH(1,1)-t, Filtered Historical Simulation

"""

import numpy as np
import pandas as pd
from scipy import stats
from arch import arch_model
import yfinance as yf
import warnings
import os
from pathlib import Path

warnings.filterwarnings("ignore")
np.random.seed(42)

In [17]:
WINDOW    = 500          # 500-day rolling lookback — standard regulatory convention
REFIT_EVERY = 5         # I refit GARCH every 5 days instead of daily.

# Daily refits on a 500-day window across ~4800 days would
# take 15+ minutes. Every-5-days is imperceptible in accuracy.

ALPHA_VAR = 0.01         # 99% VaR
ALPHA_ES  = 0.025        # 97.5% ES (FRTB standard)


OUT_DIR = Path.cwd() / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [18]:

#─────────────────────────────────────────────────────────────────────────────#
 # PHASE 1: Data & diagnostics #
#─────────────────────────────────────────────────────────────────────────────#

print("=" * 60)
print("PHASE 1: Downloading SPY data...")
print("=" * 60)
raw = yf.download("SPY", start="2005-01-01", end="2024-12-31",
                  auto_adjust=True, progress=False)

prices = raw["Close"].squeeze()

# Simple returns — cleaner intuition ("I lost 2% today") and matches regulatory convention

returns = prices.pct_change().dropna()

n_total     = len(returns)
n_forecast  = n_total - WINDOW
oos_start   = returns.index[WINDOW]
oos_end     = returns.index[-1]

print(f"\nFull return series : {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Total trading days : {n_total}")
print(f"Walk-forward start : {oos_start.date()} (day {WINDOW + 1})")
print(f"Out-of-sample days : {n_forecast}")
print(f"\nReturn diagnostics:")
print(f"  Mean daily return : {returns.mean()*100:.4f}%")
print(f"  Daily std dev     : {returns.std()*100:.4f}%")
print(f"  Skewness          : {returns.skew():.4f}")
print(f"  Excess kurtosis   : {returns.kurt():.4f}  ← fat tails confirmed")

PHASE 1: Downloading SPY data...

Full return series : 2005-01-04 → 2024-12-30
Total trading days : 5031
Walk-forward start : 2006-12-28 (day 501)
Out-of-sample days : 4531

Return diagnostics:
  Mean daily return : 0.0462%
  Daily std dev     : 1.1992%
  Skewness          : -0.0805
  Excess kurtosis   : 14.9475  ← fat tails confirmed


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Helper functions for the three models
# ─────────────────────────────────────────────────────────────────────────────

def hs_var_es(window_rets, alpha_var=ALPHA_VAR, alpha_es=ALPHA_ES):
    """Historical Simulation: empirical quantiles, nothing fancy."""
    var = -np.percentile(window_rets, alpha_var * 100)
    
    # ES = mean of returns worse than the VaR threshold
    q_es = np.percentile(window_rets, ALPHA_ES * 100)
    tail = window_rets[window_rets <= q_es]
    es = -tail.mean() if len(tail) > 0 else -q_es

    return var, es

def fit_garch_t(window_rets):
    """
    Fit GARCH(1,1) with Student-t errors. I scale returns by 100 because
    arch works better with percentage-scale inputs.
    Returns the fitted result object.
    """
    am = arch_model(window_rets * 100, vol="Garch", p=1, q=1,
                    dist="t", mean="Zero")
    res = am.fit(disp="off", show_warning=False)

    return res

def garch_var_es(garch_res, alpha_var=ALPHA_VAR, alpha_es=ALPHA_ES):

    """
    Given a fitted GARCH result, forecast next-day sigma and compute
    VaR/ES analytically using the Student-t distribution.
    The analytical ES formula for t distribution:
      ES = sigma * [ t_pdf(t_ppf(alpha), nu) / alpha ] * (nu + t_ppf(alpha)^2) / (nu - 1)
    """

    forecast   = garch_res.forecast(horizon=1, reindex=False)
    sigma_next = np.sqrt(forecast.variance.values[-1, 0]) / 100  # back to decimal scale
    nu  = garch_res.params.get("nu", 10)  # degrees of freedom
    nu  = max(nu, 2.01)                   # guard against degenerate nu
    q_var = stats.t.ppf(alpha_var, df=nu)
    q_es  = stats.t.ppf(alpha_es, df=nu)
    var = -sigma_next * q_var

    # Analytical ES for Student-t
    es_num = stats.t.pdf(q_es, df=nu) * (nu + q_es**2) / (nu - 1)
    es = sigma_next * es_num / alpha_es
    
    return var, es


def fhs_var_es(garch_res, window_rets, alpha_var=ALPHA_VAR, alpha_es=ALPHA_ES):
    """
    Filtered Historical Simulation (Hull-White):
    1. Extract standardised residuals from the GARCH fit.
    2. Take empirical quantiles of those residuals.
    3. Scale by next-day sigma forecast.
    The key insight: we keep the real shape of the tail (non-normal)
    but let GARCH handle the time-varying scale.
    """
    forecast   = garch_res.forecast(horizon=1, reindex=False)
    sigma_next = np.sqrt(forecast.variance.values[-1, 0]) / 100
    cond_vol = garch_res.conditional_volatility / 100   # fitted in-sample vol
    std_resids = (window_rets.values) / cond_vol.values # z_t = r_t / sigma_t
    var = -np.percentile(std_resids, alpha_var * 100) * sigma_next
    tail = std_resids[std_resids <= np.percentile(std_resids, alpha_es * 100)]
    es   = -tail.mean() * sigma_next if len(tail) > 0 else var
    
    return var, es





In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Walk-forward loop
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 2: Running walk-forward engine...")
print(f"  Models: HS | GARCH(1,1)-t | FHS")
print(f"  GARCH refit every {REFIT_EVERY} days")
print("=" * 60)
records     = []
garch_cache = None   # I cache the last GARCH fit and reuse it for REFIT_EVERY days

for i in range(n_forecast):
    t = WINDOW + i
    window_rets  = returns.iloc[t - WINDOW : t]
    actual_ret   = returns.iloc[t]
    date         = returns.index[t]

    # ── Historical Simulation ──────────────────────────────────────────────
    hs_var, hs_es = hs_var_es(window_rets)
    # ── GARCH-t + FHS: refit every REFIT_EVERY days ───────────────────────
    if i % REFIT_EVERY == 0 or garch_cache is None:
        garch_cache = fit_garch_t(window_rets)
    garch_var, garch_es = garch_var_es(garch_cache)
    fhs_var,   fhs_es   = fhs_var_es(garch_cache, window_rets)

    # ── Breach flags ──────────────────────────────────────────────────────
    hs_breach    = int(actual_ret < -hs_var)
    garch_breach = int(actual_ret < -garch_var)
    fhs_breach   = int(actual_ret < -fhs_var)
    records.append({
        "date":         date,
        "actual_ret":   actual_ret,
        "HS_VaR99":     hs_var,
        "HS_ES975":     hs_es,
        "GARCH_VaR99":  garch_var,
        "GARCH_ES975":  garch_es,
        "FHS_VaR99":    fhs_var,
        "FHS_ES975":    fhs_es,
        "HS_breach":    hs_breach,
        "GARCH_breach": garch_breach,
        "FHS_breach":   fhs_breach,
    })

    if (i + 1) % 250 == 0:
        print(f"  → {i+1:>4d} / {n_forecast} days done  |  date: {date.date()}")
print(f"  → {n_forecast} / {n_forecast} days done  ✓")
# ─────────────────────────────────────────────────────────────────────────────
# Save results
# ─────────────────────────────────────────────────────────────────────────────

results = pd.DataFrame(records).set_index("date")
out_path = os.path.join(OUT_DIR, "results.csv")
results.to_csv(out_path)
print(f"\nResults saved → {out_path}")
print("\nFirst 5 rows:")
print(results.head().to_string())
print("\nOverall breach counts (99% VaR):")
N = len(results)
for model in ["HS", "GARCH", "FHS"]:
    col  = f"{model}_breach"
    cnt  = results[col].sum()
    rate = cnt / N * 100
    print(f"  {model:<6}: {cnt:>3} breaches / {N} days  ({rate:.2f}%)  — expected ~{N*0.01:.0f}")




PHASE 2: Running walk-forward engine...
  Models: HS | GARCH(1,1)-t | FHS
  GARCH refit every 5 days
  →  250 / 4531 days done  |  date: 2007-12-26
  →  500 / 4531 days done  |  date: 2008-12-22
  →  750 / 4531 days done  |  date: 2009-12-18
  → 1000 / 4531 days done  |  date: 2010-12-16
  → 1250 / 4531 days done  |  date: 2011-12-13
  → 1500 / 4531 days done  |  date: 2012-12-12
  → 1750 / 4531 days done  |  date: 2013-12-10
  → 2000 / 4531 days done  |  date: 2014-12-08
  → 2250 / 4531 days done  |  date: 2015-12-04
  → 2500 / 4531 days done  |  date: 2016-12-01
  → 2750 / 4531 days done  |  date: 2017-11-29
  → 3000 / 4531 days done  |  date: 2018-11-27
  → 3250 / 4531 days done  |  date: 2019-11-25
  → 3500 / 4531 days done  |  date: 2020-11-20
  → 3750 / 4531 days done  |  date: 2021-11-18
  → 4000 / 4531 days done  |  date: 2022-11-16
  → 4250 / 4531 days done  |  date: 2023-11-15
  → 4500 / 4531 days done  |  date: 2024-11-13
  → 4531 / 4531 days done  ✓

Results saved → c:\Use

In [21]:
"""
Phase 3: Backtesting suite — Kupiec POF, Christoffersen independence,
and Basel Traffic Light zones.
"""

import numpy as np
import pandas as pd
from scipy import stats
import os


OUT_DIR = Path.cwd() / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Load results
# ─────────────────────────────────────────────────────────────────────────────

results = pd.read_csv(os.path.join(OUT_DIR, "results.csv"),
                      index_col="date", parse_dates=True)
N = len(results)
print(f"Loaded {N} out-of-sample days  ({results.index[0].date()} → {results.index[-1].date()})\n")

ALPHA = 0.01   # 99% VaR target breach rate


# ─────────────────────────────────────────────────────────────────────────────
# 1. Kupiec POF Test (unconditional coverage)
# ─────────────────────────────────────────────────────────────────────────────

def kupiec_pof(breach_series, alpha=ALPHA):
    """
    LR_uc = -2 * [ x*log(alpha) + (N-x)*log(1-alpha)
                   - x*log(x/N) - (N-x)*log(1-x/N) ]
    H0: empirical breach rate == alpha
    """
    x     = int(breach_series.sum())
    n     = len(breach_series)
    p_hat = x / n

    if x == 0 or x == n:
        return np.nan, np.nan  # edge case: no breaches or all breaches

    lr = -2 * (
        x * np.log(alpha)     + (n - x) * np.log(1 - alpha)
        - x * np.log(p_hat)   - (n - x) * np.log(1 - p_hat)
    )
    p_val = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_val


# ─────────────────────────────────────────────────────────────────────────────
# 2. Christoffersen Independence Test (breach clustering)
# ─────────────────────────────────────────────────────────────────────────────

def christoffersen_ind(breach_series):
    """
    Build 2x2 transition matrix of consecutive breach pairs.
    n_ij = count of (state_i at t → state_j at t+1).
    LR_ind tests whether P(breach|prev breach) == P(breach|prev no breach).
    """
    b = breach_series.values.astype(int)
    n00 = ((b[:-1] == 0) & (b[1:] == 0)).sum()
    n01 = ((b[:-1] == 0) & (b[1:] == 1)).sum()
    n10 = ((b[:-1] == 1) & (b[1:] == 0)).sum()
    n11 = ((b[:-1] == 1) & (b[1:] == 1)).sum()

    # Transition probabilities
    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi   = (n01 + n11) / (n00 + n01 + n10 + n11)

    # Guard against log(0)
    def safe_log(x):
        return np.log(x) if x > 0 else -1e10

    ll_h0 = (n01 + n11) * safe_log(pi) + (n00 + n10) * safe_log(1 - pi)
    ll_h1 = (n00 * safe_log(1 - pi01) + n01 * safe_log(pi01) +
              n10 * safe_log(1 - pi11) + n11 * safe_log(pi11))

    lr = -2 * (ll_h0 - ll_h1)
    p_val = 1 - stats.chi2.cdf(lr, df=1)

    return lr, p_val, pi01, pi11


# ─────────────────────────────────────────────────────────────────────────────
# 3. Basel Traffic Light (rolling 250-day window)
# ─────────────────────────────────────────────────────────────────────────────

BASEL_WINDOW = 250

def classify_zone(count):
    if count <= 4:
        return "Green"
    elif count <= 9:
        return "Yellow"
    else:
        return "Red"


def traffic_light(breach_series):
    rolling_counts = breach_series.rolling(BASEL_WINDOW, min_periods=BASEL_WINDOW).sum()
    zones = rolling_counts.dropna().apply(classify_zone)
    pct = zones.value_counts(normalize=True) * 100
    return rolling_counts, zones, pct


# ─────────────────────────────────────────────────────────────────────────────
# 4. Stress window analysis helper
# ─────────────────────────────────────────────────────────────────────────────

STRESS_WINDOWS = {
    "GFC 2008":     ("2008-09-01", "2009-03-31"),
    "COVID 2020":   ("2020-02-01", "2020-04-30"),
    "Rates 2022":   ("2022-01-01", "2022-12-31"),
}


def stress_breach_counts(results_df, start, end):
    sub = results_df.loc[start:end]
    return {
        "HS":    int(sub["HS_breach"].sum()),
        "GARCH": int(sub["GARCH_breach"].sum()),
        "FHS":   int(sub["FHS_breach"].sum()),
        "days":  len(sub),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 5. Run everything and build summary
# ─────────────────────────────────────────────────────────────────────────────

models   = ["HS", "GARCH", "FHS"]
summary  = {}

print("=" * 70)
print("KUPIEC POF TEST (99% VaR)")
print("=" * 70)
print(f"{'Model':<8} {'Expected':>10} {'Actual':>8} {'Rate%':>8} {'LR':>8} {'p-val':>8} {'Pass?':>6}")
print("-" * 70)

for m in models:
    col         = f"{m}_breach"
    x           = int(results[col].sum())
    expected    = N * ALPHA
    rate        = x / N * 100
    lr, p       = kupiec_pof(results[col])
    passed      = "✓" if (p is not None and p > 0.05) else "✗"
    print(f"{m:<8} {expected:>10.1f} {x:>8d} {rate:>7.2f}% {lr:>8.3f} {p:>8.4f} {passed:>6}")
    summary[m] = {
        "N": N, "Breaches": x, "HitRate_pct": round(rate, 3),
        "Kupiec_LR": round(lr, 4), "Kupiec_p": round(p, 4),
        "Kupiec_pass": p > 0.05,
    }

print()
print("=" * 70)
print("CHRISTOFFERSEN INDEPENDENCE TEST")
print("=" * 70)
print(f"{'Model':<8} {'LR':>8} {'p-val':>8} {'P(B|prev B)':>12} {'P(B|prev 0)':>12} {'Pass?':>6}")
print("-" * 70)

for m in models:
    col             = f"{m}_breach"
    lr, p, pi11, pi01 = christoffersen_ind(results[col])
    passed          = "✓" if p > 0.05 else "✗"
    print(f"{m:<8} {lr:>8.3f} {p:>8.4f} {pi11:>12.4f} {pi01:>12.4f} {passed:>6}")
    summary[m].update({
        "Christ_LR": round(lr, 4), "Christ_p": round(p, 4),
        "Christ_pass": p > 0.05,
        "P_breach_given_breach": round(pi11, 4),
        "P_breach_given_no_breach": round(pi01, 4),
    })

print()
print("=" * 70)
print("BASEL TRAFFIC LIGHT ZONES (rolling 250-day window)")
print("=" * 70)
print(f"{'Model':<8} {'% Green':>10} {'% Yellow':>10} {'% Red':>8}")
print("-" * 70)

traffic_data = {}
for m in models:
    col                      = f"{m}_breach"
    rolling_counts, zones, pct = traffic_light(results[col])
    traffic_data[m]          = {"rolling_counts": rolling_counts, "zones": zones}
    g = pct.get("Green",  0)
    y = pct.get("Yellow", 0)
    r = pct.get("Red",    0)
    print(f"{m:<8} {g:>9.1f}% {y:>9.1f}% {r:>7.1f}%")
    summary[m].update({
        "PctGreen": round(g, 2), "PctYellow": round(y, 2), "PctRed": round(r, 2)
    })

print()
print("=" * 70)
print("STRESS PERIOD BREACH COUNTS")
print("=" * 70)
print(f"{'Period':<18} {'Days':>6} {'HS':>6} {'GARCH':>7} {'FHS':>6}")
print("-" * 70)

for name, (s, e) in STRESS_WINDOWS.items():
    bc = stress_breach_counts(results, s, e)
    print(f"{name:<18} {bc['days']:>6} {bc['HS']:>6} {bc['GARCH']:>7} {bc['FHS']:>6}")
    for m in models:
        summary[m][f"Breach_{name.replace(' ','_')}"] = bc[m]

# ─────────────────────────────────────────────────────────────────────────────
# Save summary + rolling breach counts (needed for plots)
# ─────────────────────────────────────────────────────────────────────────────

summary_df = pd.DataFrame(summary).T
summary_df.to_csv(os.path.join(OUT_DIR, "backtest_summary.csv"))
print(f"\nSummary saved → {os.path.join(OUT_DIR, 'backtest_summary.csv')}")

# Save rolling breach counts so phase4 can plot Basel zones without re-running
for m in models:
    traffic_data[m]["rolling_counts"].to_csv(
        os.path.join(OUT_DIR, f"rolling_counts_{m}.csv"), header=True
    )

print("\nFull summary:")
print(summary_df.to_string())


Loaded 4531 out-of-sample days  (2006-12-28 → 2024-12-30)

KUPIEC POF TEST (99% VaR)
Model      Expected   Actual    Rate%       LR    p-val  Pass?
----------------------------------------------------------------------
HS             45.3       80    1.77%   21.849   0.0000      ✗
GARCH          45.3       40    0.88%    0.654   0.4185      ✓
FHS            45.3       55    1.21%    1.960   0.1616      ✓

CHRISTOFFERSEN INDEPENDENCE TEST
Model          LR    p-val  P(B|prev B)  P(B|prev 0)  Pass?
----------------------------------------------------------------------
HS         19.683   0.0000       0.0160       0.1125      ✗
GARCH      12.818   0.0003       0.0080       0.1000      ✗
FHS        16.771   0.0000       0.0109       0.1091      ✗

BASEL TRAFFIC LIGHT ZONES (rolling 250-day window)
Model       % Green   % Yellow    % Red
----------------------------------------------------------------------
HS            58.5%      25.5%    16.0%
GARCH         92.9%       7.1%     0.0%
FHS 

In [22]:
"""
Phase 4: All publication-quality visualisations for the blog post.
Needs results.csv and rolling_counts_*.csv from phase3.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import os, warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

OUT_DIR = Path.cwd() / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR  = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

COLORS = {"HS": "#e63946", "GARCH": "#2a9d8f", "FHS": "#457b9d"}
STRESS = {
    "GFC 2008":   ("2008-09-01", "2009-03-31"),
    "COVID 2020": ("2020-02-01", "2020-04-30"),
    "Rates 2022": ("2022-01-01", "2022-12-31"),
}

# ─────────────────────────────────────────────────────────────────────────────
results = pd.read_csv(os.path.join(OUT_DIR, "results.csv"),
                      index_col="date", parse_dates=True)
rc = {m: pd.read_csv(os.path.join(OUT_DIR, f"rolling_counts_{m}.csv"),
                     index_col=0, parse_dates=True).squeeze()
      for m in ["HS", "GARCH", "FHS"]}

print("Loaded data. Generating plots...")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1: Full-sample breach clustering (3 stacked subplots)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle("99% VaR Breach Clustering: Static vs Dynamic Models (2007–2024)",
             fontsize=14, fontweight="bold", y=1.01)

for ax, m in zip(axes, ["HS", "GARCH", "FHS"]):
    var_col = f"{m}_VaR99"
    br_col  = f"{m}_breach"
    color   = COLORS[m]

    ax.bar(results.index, results["actual_ret"] * 100,
           color="#adb5bd", width=1, alpha=0.6, label="Actual return")
    ax.plot(results.index, -results[var_col] * 100,
            color=color, lw=1.2, label=f"{m} 99% VaR")

    breaches = results[results[br_col] == 1]
    ax.scatter(breaches.index, breaches["actual_ret"] * 100,
               color="#d62828", s=18, zorder=5, label="Breach")

    # Shade the three stress windows
    for name, (s, e) in STRESS.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.08, color="orange")

    n_br = int(results[br_col].sum())
    ax.set_ylabel("Daily Return (%)", fontsize=9)
    ax.set_title(f"{m}  ({n_br} total breaches)", fontsize=11, color=color)
    ax.legend(fontsize=8, loc="lower left")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
p1 = os.path.join(PLOT_DIR, "breach_clustering_full_sample.png")
fig.savefig(p1, dpi=200, bbox_inches="tight")
plt.close()
print(f"  ✓ {p1}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2: Basel Traffic Light — rolling 250-day breach counts
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_title("Basel Traffic Light: Rolling 250-Day Breach Counts (99% VaR)",
             fontsize=13, fontweight="bold")

# Zone background bands
ax.axhspan(0,  4.5, alpha=0.12, color="green",  label="Green zone (0–4)")
ax.axhspan(4.5, 9.5, alpha=0.12, color="yellow", label="Yellow zone (5–9)")
ax.axhspan(9.5, 25,  alpha=0.12, color="red",    label="Red zone (10+)")
ax.axhline(4.5, color="green",  lw=0.8, ls="--", alpha=0.5)
ax.axhline(9.5, color="orange", lw=0.8, ls="--", alpha=0.5)

for m, series in rc.items():
    ax.plot(series.index, series.values, lw=1.5,
            color=COLORS[m], label=m, alpha=0.9)

# Stress annotations
stress_colors = {"GFC 2008": "#ffb347", "COVID 2020": "#87ceeb", "Rates 2022": "#dda0dd"}
for name, (s, e) in STRESS.items():
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
               alpha=0.18, color=stress_colors[name])
    mid = pd.Timestamp(s) + (pd.Timestamp(e) - pd.Timestamp(s)) / 2
    ax.text(mid, 20.5, name, ha="center", fontsize=8, color="#444")

ax.set_ylabel("Breach count in last 250 days", fontsize=10)
ax.set_ylim(0, 24)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
p2 = os.path.join(PLOT_DIR, "basel_traffic_light_zones.png")
fig.savefig(p2, dpi=200, bbox_inches="tight")
plt.close()
print(f"  ✓ {p2}")

# ─────────────────────────────────────────────────────────────────────────────
# Helper: deep-dive plot for any stress window
# ─────────────────────────────────────────────────────────────────────────────

def stress_deep_dive(start, end, title, event_date, event_label, fname):
    sub  = results.loc[start:end].copy()
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
    fig.suptitle(title, fontsize=13, fontweight="bold")

    ax = axes[0]
    ax.bar(sub.index, sub["actual_ret"] * 100,
           color="#adb5bd", width=1, alpha=0.7, label="Actual return")
    for m in ["HS", "GARCH", "FHS"]:
        ax.plot(sub.index, -sub[f"{m}_VaR99"] * 100,
                color=COLORS[m], lw=1.5, label=f"{m} 99% VaR")
        breaches = sub[sub[f"{m}_breach"] == 1]
        ax.scatter(breaches.index, breaches["actual_ret"] * 100,
                   color=COLORS[m], marker="v", s=60, zorder=5)

    ax.axvline(pd.Timestamp(event_date), color="#d62828",
               lw=1.5, ls="--", alpha=0.8)
    ax.text(pd.Timestamp(event_date), ax.get_ylim()[0] * 0.92,
            f" {event_label}", color="#d62828", fontsize=8, va="bottom")
    ax.set_ylabel("Daily Return (%)", fontsize=10)
    ax.legend(fontsize=9, loc="lower left")
    ax.set_title("Daily returns vs 99% VaR (▼ = breach per model colour)", fontsize=10)

    # Bottom panel: cumulative return
    cum = (1 + sub["actual_ret"]).cumprod() - 1
    axes[1].fill_between(cum.index, cum * 100, 0,
                         where=cum >= 0, alpha=0.5, color="#2a9d8f")
    axes[1].fill_between(cum.index, cum * 100, 0,
                         where=cum < 0, alpha=0.5, color="#e63946")
    axes[1].plot(cum.index, cum * 100, color="#222", lw=1)
    axes[1].set_ylabel("Cumulative Return (%)", fontsize=9)
    axes[1].axhline(0, color="#555", lw=0.8)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    path = os.path.join(PLOT_DIR, fname)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {path}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS 3, 4, 5: Stress deep dives
# ─────────────────────────────────────────────────────────────────────────────

stress_deep_dive("2020-01-01", "2020-06-30",
                 "Deep Dive: March 2020 COVID Crash",
                 "2020-03-11", "WHO Declares Pandemic",
                 "covid_march2020_deep_dive.png")

stress_deep_dive("2008-07-01", "2009-04-30",
                 "Deep Dive: 2008 Global Financial Crisis",
                 "2008-09-15", "Lehman Collapses",
                 "gfc_2008_deep_dive.png")

stress_deep_dive("2021-11-01", "2023-01-31",
                 "Deep Dive: 2022 Inflation & Rate-Hike Cycle",
                 "2022-03-16", "First Fed Rate Hike",
                 "rate_hike_2022.png")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 6: VaR distribution comparison
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle("Distribution of Daily 99% VaR Estimates vs Actual Returns",
             fontsize=13, fontweight="bold")

for ax, m in zip(axes, ["HS", "GARCH", "FHS"]):
    var_col = f"{m}_VaR99"
    ax.hist(results["actual_ret"] * 100, bins=120, density=True,
            color="#adb5bd", alpha=0.5, label="Actual returns")
    ax.hist(-results[var_col] * 100, bins=80, density=True,
            color=COLORS[m], alpha=0.65, label=f"{m} VaR (negated)")
    ax.set_title(m, fontsize=11, color=COLORS[m])
    ax.set_xlabel("Daily Return / VaR (%)", fontsize=9)
    ax.legend(fontsize=8)
    ax.axvline(0, color="#555", lw=0.7, ls="--")

axes[0].set_ylabel("Density", fontsize=10)
plt.tight_layout()
p6 = os.path.join(PLOT_DIR, "var_es_comparison_distributions.png")
fig.savefig(p6, dpi=200, bbox_inches="tight")
plt.close()
print(f"  ✓ {p6}")

print("\nAll 6 plots saved to:", PLOT_DIR)


Loaded data. Generating plots...
  ✓ c:\Users\uvsha\Downloads\output\plots\breach_clustering_full_sample.png
  ✓ c:\Users\uvsha\Downloads\output\plots\basel_traffic_light_zones.png
  ✓ c:\Users\uvsha\Downloads\output\plots\covid_march2020_deep_dive.png
  ✓ c:\Users\uvsha\Downloads\output\plots\gfc_2008_deep_dive.png
  ✓ c:\Users\uvsha\Downloads\output\plots\rate_hike_2022.png
  ✓ c:\Users\uvsha\Downloads\output\plots\var_es_comparison_distributions.png

All 6 plots saved to: c:\Users\uvsha\Downloads\output\plots
